# RoSEHFL — ResNet-18 / CIFAR-100 (Geant2010, 30 nodes)

Third dataset-model pair from the ShapeFL paper (Deng et al., IEEE/ACM ToN 2024).

In [ ]:
GITHUB_OWNER = 'SakiburRahman07'
GITHUB_REPO = 'RoSEHFL'
GITHUB_BRANCH = 'main'

RUN_NAME = 'cmp_8_resnet18_cifar100_geant2010_n30_seed42'

CHECKPOINT_EVERY = 10     # Flower rounds between full checkpoint writes
SYNC_MIN_INTERVAL = 120   # seconds between rclone pushes

# Clients train on GPU. On CPU this pairing measured 111 s per client
# local epoch (769 h for all 8 strategies); on GPU ~1.3 s (~20 h).
CLIENT_GPUS = 0.5   # Ray GPU fraction per client; with 2 GPUs -> actors spread over both
CLIENT_CPUS = 1.0   # pool = min(vCPU/CLIENT_CPUS, nGPU/CLIENT_GPUS)


In [ ]:
!apt-get update -y -qq
!apt-get install -y -qq rclone git

from pathlib import Path
WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'RoSEHFL'

import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'free disk: {free_gb:.1f} GB')
assert free_gb > 15, 'Need headroom: the rosehfl checkpoint is ~3.6 GB and is written atomically (2x transient).'

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
RCLONE_CONF_TEXT = secrets.get_secret('RCLONE_CONF')
GDRIVE_FOLDER_ID = secrets.get_secret('GDRIVE_FOLDER_ID')
print('Secrets loaded.')

In [ ]:
if REPO_DIR.exists():
    !rm -rf /kaggle/working/RoSEHFL

clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
!git clone --branch {GITHUB_BRANCH} --single-branch {clone_url} /kaggle/working/RoSEHFL
%cd /kaggle/working/RoSEHFL
!git rev-parse --short HEAD

In [ ]:
!python -m pip install -q --upgrade pip
!pip install -q -r requirements.txt

In [ ]:
cfg_dir = Path('/root/.config/rclone')
cfg_dir.mkdir(parents=True, exist_ok=True)
(cfg_dir / 'rclone.conf').write_text(RCLONE_CONF_TEXT.replace('\\n', '\n'), encoding='utf-8')
!rclone lsd gdrive: | head

In [ ]:
import os, subprocess

DATASET_CACHE_LOCAL = Path('/kaggle/working/RoSEHFL/dataset')
DATASET_CACHE_REMOTE = 'gdrive:RoSEHFL/dataset_cache'
r = subprocess.run(['rclone', 'copy', DATASET_CACHE_REMOTE, str(DATASET_CACHE_LOCAL),
                    '--transfers', '8', '--checkers', '8'], capture_output=True, text=True)
print('dataset cache pull:', 'OK' if r.returncode == 0 else 'FAILED (CIFAR-100 downloads fresh, ~169 MB)')
os.environ['ROSEHFL_DATA_DIR'] = str(DATASET_CACHE_LOCAL)

## Partition sanity check

CIFAR-100 has almost no shard slack (100 classes x 33 shards = 3300 against 30 x 100 = 3000
demanded). Confirm every node gets its full 1500 samples **before** spending hours training.

In [ ]:
import warnings, io, contextlib
import numpy as np
from src.data.data_loader import load_data, create_non_iid_partitions, DATASET_INFO

train_ds, _ = load_data('cifar100', augment=False)
info = DATASET_INFO['cifar100']
print('paper spec ->', {k: info[k] for k in ('shards_per_node', 'classes_per_node')})

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    with contextlib.redirect_stdout(io.StringIO()):
        parts = create_non_iid_partitions(
            train_ds, 30, 15, info['shards_per_node'], info['classes_per_node'], 42)

sizes = np.array([len(parts[n]) for n in range(30)])
labels = np.asarray(train_ds.targets)
ncls = [len(set(labels[parts[n]].tolist())) for n in range(30)]
print(f'samples/node: min={sizes.min()} max={sizes.max()} total={sizes.sum():,}')
print(f'classes/node: min={min(ncls)} max={max(ncls)} (expected 20)')
for w in caught:
    print('WARNING:', w.message)
assert sizes.min() == 1500 and not caught, '[ERROR] Partition starvation'
print('\nOK: all 30 nodes have the full 1500 samples across 20 classes.')

In [ ]:
LOCAL_RUN_DIR = Path('/kaggle/working/RoSEHFL/results') / RUN_NAME
REMOTE_RUN_DIR = f'gdrive:RoSEHFL/kaggle_runs/{RUN_NAME}'
LOCAL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)

r = subprocess.run(['rclone', 'copy', REMOTE_RUN_DIR, str(LOCAL_RUN_DIR),
                    '--create-empty-src-dirs', '--transfers', '8', '--checkers', '8'],
                   capture_output=True, text=True)
print('previous-state pull:', 'OK' if r.returncode == 0 else 'none found (first run)')

os.environ['ROSEHFL_SYNC_LOCAL'] = str(LOCAL_RUN_DIR)
os.environ['ROSEHFL_SYNC_REMOTE'] = REMOTE_RUN_DIR
os.environ['ROSEHFL_SYNC_MIN_INTERVAL'] = str(SYNC_MIN_INTERVAL)
os.environ['ROSEHFL_CHECKPOINT_EVERY'] = str(CHECKPOINT_EVERY)
print(f'checkpoint every {CHECKPOINT_EVERY} rounds; sync at most every {SYNC_MIN_INTERVAL}s')

import json
sp = LOCAL_RUN_DIR / 'run_status.json'
if sp.exists():
    print('completed so far:', json.loads(sp.read_text())['completed_strategies'])
os.environ['ROSEHFL_CLIENT_GPUS'] = str(CLIENT_GPUS)
os.environ['ROSEHFL_CLIENT_CPUS'] = str(CLIENT_CPUS)
print(f'clients: {CLIENT_CPUS} cpu + {CLIENT_GPUS} gpu each')


## GPU check

Confirm the GPU is visible and the actor pool sizes as expected **before** training.


In [ ]:
import os
import torch

n = torch.cuda.device_count()
print('GPUs visible:', n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print('  [%d] %s  %.1f GB' % (i, p.name, p.total_memory / 1e9))
assert n >= 1, ('No GPU selected. Pick T4 x2 or P100 in Kaggle settings, '
                'otherwise clients fall back to CPU: ~769 h for all 8 strategies.')

# Flower sizes the pool as min(vCPU/CLIENT_CPUS, nGPU/CLIENT_GPUS)
# -> flwr/simulation/ray_transport/ray_actor.py::pool_size_from_resources
vcpu = os.cpu_count()
by_cpu = int(vcpu / CLIENT_CPUS)
by_gpu = int(n / CLIENT_GPUS)
pool = min(by_cpu, by_gpu)
print('')
print('vCPU=%d -> actor pool = min(%d, %d) = %d concurrent clients'
      % (vcpu, by_cpu, by_gpu, pool))
print('spread over %d GPU(s) => about %.1f client(s) per card' % (n, pool / n))
assert pool >= 1, 'Actor pool would be empty - lower CLIENT_CPUS or CLIENT_GPUS.'

# ResNet-18 @ batch 32 measured ~290 MB reserved, plus CUDA context (~300-500 MB)
per_actor_gb = 0.8
need = per_actor_gb * (pool / n)
have = torch.cuda.get_device_properties(0).total_memory / 1e9
print('')
print('est %.1f GB needed per card, %.1f GB available -> %s'
      % (need, have, 'OK' if need < have * 0.85 else 'TIGHT: lower CLIENT_GPUS'))


In [ ]:
!python -m scripts.run_comparison --resume \
  --strategies fedavg fedprox cost_first data_first random share shapefl rosehfl \
  --model resnet18 --dataset cifar100 --topology geant2010 --num-nodes 30 \
  --kappa-e 1 --kappa-c 10 --kappa 50 --gamma-max 2800 --probe-size 1000 \
  --seed 42 --comparison-mode effective --output-dir {LOCAL_RUN_DIR}

## Health check

In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
from src.models.factory import get_model
from src.data.data_loader import load_data

sd = LOCAL_RUN_DIR / 'strategies' / 'rosehfl'
ck = pickle.load(open(sd / 'checkpoint.pkl', 'rb'))
print('cloud_round =', ck['cloud_round'], ' flower_rounds =', ck['completed_flower_rounds'])
print('checkpoint size = %.2f GB' % ((sd / 'checkpoint.pkl').stat().st_size / 1e9))

m = get_model('resnet18', 100, 3, 'cpu')
keys = list(m.state_dict().keys())
m.load_state_dict({k: torch.tensor(np.asarray(v))
                   for k, v in zip(keys, ck['global_parameters'])}, strict=True)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_, test = load_data('cifar100', augment=False)
loader = torch.utils.data.DataLoader(torch.utils.data.Subset(test, list(range(2000))),
                                     batch_size=256, shuffle=False)
m.eval().to(dev)
crit = nn.CrossEntropyLoss(); tl = c = n = 0; spread = None; preds = set()
with torch.no_grad():
    for x, y in loader:
        x, y = x.to(dev), y.to(dev); o = m(x)
        if spread is None: spread = float(o.std(dim=0).mean())
        tl += crit(o, y).item() * y.size(0); p = o.argmax(1)
        c += (p == y).sum().item(); n += y.size(0); preds |= set(p.cpu().numpy().tolist())

rm_zero = sum(1 for i, k in enumerate(keys)
              if 'running_mean' in k and np.all(np.asarray(ck['global_parameters'][i]) == 0))
print(f'loss={tl/n:.4f}  (ln100={np.log(100):.4f})   acc={c/n:.4f}  (chance=0.01)')
print(f'logit spread   = {spread:.3e}   (healthy >1e-3)')
print(f'distinct preds = {len(preds)}/100')
print(f'BN running_mean still at init: {rm_zero}/20')

ok = spread > 1e-3 and len(preds) > 1 and rm_zero == 0
print('\nVERDICT:', 'HEALTHY' if ok else 'BROKEN - stop and investigate')

In [ ]:
print('Final sync to Drive...')
!rclone copy {LOCAL_RUN_DIR} {REMOTE_RUN_DIR} --create-empty-src-dirs --transfers 4 --checkers 4 --progress
!rclone copy {DATASET_CACHE_LOCAL} {DATASET_CACHE_REMOTE} --transfers 8 --checkers 8
print('Done.')